# ДТП в Нижегородской области

saintson1, [отчёт](../solution/analysis.md), запуск дашборда: `make run`

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'solution/app.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'solution'))

import charts
from data_loading import load_records, records_to_frame
from projections import pca_figure, tsne_figure
from stats import run_hypotheses

SOURCE = ROOT / 'data/raw/nizhegorodskaia-oblast.geojson'


## Данные

Я взял одно ДТП за одно наблюдение. Получил выгрузку с 01.01.2015 по 31.01.2026 командой `.venv/bin/python data/download_data.py`.

In [ ]:
frame = records_to_frame(load_records(SOURCE))
source = json.loads(SOURCE.with_suffix('.source.json').read_text())
print('Регион:', source['region'])
print('SHA-256:', source['sha256'])
print('Получено:', source['downloaded_at_utc'])
print('Размер:', frame.shape)
print('Период:', frame.datetime.min(), '-', frame.datetime.max())
print('Повторные ID:', frame.id.duplicated().sum())
display(frame[['datetime', 'region', 'category', 'participants_count', 'injured_count', 'dead_count']].head())


## Проверка качества

Я исправил 373 пары координат. Остальные сомнительные координаты исключил только из карт.

In [ ]:
columns = ['datetime', 'region', 'category', 'light', 'participants_count',
           'injured_count', 'dead_count', 'vehicles_count', 'longitude_raw', 'latitude_raw']
display(frame[columns].isna().sum().rename('missing').to_frame())
assert frame.source_row.is_unique
assert (frame[['participants_count', 'injured_count', 'dead_count', 'vehicles_count']] >= 0).all().all()
assert (frame.injured_count + frame.dead_count <= frame.participants_count).all()
print('Исправлены пары:', int(frame.coordinates_swapped.sum()))
print('Карта:', int(frame.map_valid.sum()), 'из', len(frame))
display(frame[['participants_count', 'injured_count', 'dead_count', 'vehicles_count']].describe().T)


## Обзор

Я построил графики 1-4. В 2026 году есть только январь. Карту часов я нормировал внутри дня недели.

In [ ]:
start, end = frame.datetime.min().date(), frame.datetime.max().date()
for figure in [charts.monthly(frame, start, end), charts.categories(frame),
               charts.injured_ecdf(frame), charts.hour_weekday(frame)]:
    display(figure)


## Вопросы

На графиках 5-8 я сравнил ночную долю, затем число раненых в ночных столкновениях. Отдельно сравнил сезонные доли пешеходных ДТП.

In [ ]:
complete = frame[frame.year.between(2015, 2025)]
for figure in [charts.night_by_weekday(frame), charts.night_collision_injuries(frame),
               charts.pedestrian_seasons(complete), charts.participants_injured(frame)]:
    display(figure)


## Гипотезы

Я оставил 2015-2025 годы. Использовал двусторонние тесты и уровень значимости 0,05. В H0 предположил равенство сравниваемых долей или средних, в H1 предположил различие. Эффект считал как первую группу минус вторую. Для 95% доверительных интервалов сделал 5000 bootstrap-повторов целых ДТП. Для перестановочных тестов сделал 9999 повторов, p=(b+1)/10000.

In [ ]:
hypotheses = run_hypotheses(frame)
display(pd.DataFrame(hypotheses['results'])[['id', 'question', 'n_a', 'n_b', 'test',
                                          'mean_a', 'mean_b', 'effect', 'unit',
                                          'ci_low', 'ci_high', 'p_value', 'reject']])


Я не подтвердил различие среднего числа раненых: p=0,174, доверительный интервал включает ноль.

## Карты

Для графика 9 я взял до 5000 точек с seed=42. Для графика 10 использовал все пригодные координаты и сетку 5 на 5 км в EPSG:32638. Подложка OpenStreetMap contributors.

In [ ]:
display(charts.point_map(frame))
display(charts.grid_map(frame, step_m=5000))


## Проекции

Я взял те же 2500 ДТП для PCA и двух t-SNE, seed=42. Стандартизировал числа участников и ТС, тип и освещение закодировал OHE. Последствия использовал только для цвета.

In [ ]:
if not (ROOT / 'solution/cache/projections.csv').exists() or not (ROOT / 'solution/results.json').exists():
    from analyze import main
    main()
results = json.loads((ROOT / 'solution/results.json').read_text())
projected = dict(results['projections'], data=pd.read_csv(ROOT / 'solution/cache/projections.csv'))
metadata = projected['metadata']
print('Число преобразованных признаков:', len(metadata['features']))
print('EVR:', metadata['pca']['explained_variance_ratio'])
print('Всего сохранено:', metadata['pca']['retained_variance'])
display(pca_figure(projected))
display(pd.DataFrame(metadata['pca']['components'], columns=metadata['features'], index=['PC1', 'PC2']).T)


Я получил 68,13% сохранённой дисперсии PCA. По коэффициентам PC1 и исходным признакам связал её с масштабом ДТП.

In [ ]:
display(tsne_figure(projected))
print('Уникальных профилей X среди 2500 событий:', metadata['unique_feature_rows_sample'])
print('Параметры t-SNE:')
for perplexity, settings in metadata['tsne'].items():
    print(perplexity, {key: settings[key] for key in ['init', 'learning_rate', 'max_iter', 'random_state']})


При смене perplexity я получил другое расположение пятен. Часть плотных групп объяснил повторяющимися профилями. Расстояния между пятнами не интерпретировал буквально.

## До и после

Я сохранил частоты 18 типов ДТП, изменил сортировку, направление столбцов и подписи числа и доли.

In [ ]:
from PIL import Image
for name in ['categories_before.png', 'categories_after.png']:
    display(Image.open(ROOT / 'solution/figures' / name))


## Выводы

1. Я обнаружил более высокую долю ночных ДТП в выходные.
2. Я не подтвердил различие среднего числа раненых в ночных столкновениях.
3. Я обнаружил более высокую долю пешеходных ДТП зимой.

Дальше я бы проверил устойчивость различий внутри лет и территорий. Без данных о потоке я не могу оценить риск поездки.